# 01. The import system

**Tier:** Deep. **Why you'd use this:** A second `import` should not re-run side effects, circular imports can bite, and `reload` does not update names you already bound.

**After this lesson you can:** explain that a second `import` does not run the file again; explain a circular import; tell a reloaded module from a name you already bound.
About 30 minutes. **Uses:** [Imports](../../course/13-modules/01-imports-lesson.ipynb), [Attribute lookup](../08-attributes/01-lookup-lesson.ipynb)


## Quick review

1. What is the MRO for?
2. When does `__getattr__` run compared with a normal missing attribute?

<details><summary>Answers</summary>

The order Python searches base classes for an attribute. After normal lookup fails — it is a fallback, not the first step.
</details>


## The idea

`import` finds a module, runs its code once, and stores the module object in `sys.modules` under its name. The next `import` of that name returns the stored object and does not run the file again. A module is an object: its global names live in the module's `__dict__`, the same kind of namespace an instance uses for attributes.

`sys.path` is the list of directories searched, in order. The first entry is the directory of the program being run, which in a notebook is the notebook's folder. Later entries hold the standard library and installed packages.

<details><summary>Explain more</summary>

`from module import name` copies the current object into your namespace. It does not create a live link. `importlib.reload` runs the file again in the same module object. Names you already copied stay pointed at the old objects.

A circular import is two modules that each import the other while the first has not finished running. The half-built module is already in `sys.modules`, so the second import does not start it over. It sees whatever names have been defined so far.
</details>

<details><summary>If you already write Java, JavaScript, or C#</summary>

This cache is like a class loader that will not initialize a class twice. `reload` is the extra tool, and it does not update every variable that already took a copy of an export.
</details>


## See it work

A small rate table lives in its own file. The loader imports it once, then a later import must see the same object, including a correction written onto that object. After the file on disk changes, `reload` runs the new source in that same object. A name bound with `from rate import rate` does not follow along.


In [1]:
import importlib
import sys
import tempfile
from pathlib import Path

folder = Path(tempfile.mkdtemp())
sys.path.insert(0, str(folder))
source = folder / "rate.py"
source.write_text("rate = 10\n")
# Drop a bytecode file so reload must read the source we just wrote.
cache = folder / "__pycache__"
if cache.exists():
    for cached in cache.glob("*.pyc"):
        cached.unlink()
importlib.invalidate_caches()

rate = importlib.import_module("rate")
rate.rate = 99
rate_again = importlib.import_module("rate")
print(rate_again is rate)
print(rate_again.rate)
print(type(rate).__name__)


True
99
module


| step | what runs | `rate.rate` |
| --- | --- | --- |
| first import | `rate.py` runs, `rate = 10` | 10 |
| `rate.rate = 99` | no file runs; the attribute is overwritten | 99 |
| second import | cache hit in `sys.modules` | 99 |

```
sys.modules["rate"] --> one module object
import rate as rate_again --> same arrow
```


## Check yourself

Answer these from the example above before you run the cell.


1. Why does the second `import json` keep `marker`?
2. Why can an old `get()` see a global that reload just assigned?
3. Why is the half-built module already in `sys.modules` during a cycle?

<details><summary>Answers</summary>

1. The name `"json"` is already in `sys.modules`. The second import returns that object and does not execute `json.py`.
2. The old function's `__globals__` is the module dict. Reload re-executes into that same dict, so a global read inside the old function sees the new assignment. The function object itself is still the old one.
3. Import inserts the module object before running the file, so the other module's import can find it and avoid a second, empty copy. The cost is that the found object may not have its names yet.
</details>


In [2]:
import importlib
import sys

json = importlib.import_module("json")
json.marker = "kept"
json_again = importlib.import_module("json")
print(json_again is json)
print(json_again.marker)
print("json" in sys.modules)


True
kept
True


<details><summary>Check your answers</summary>

- `json_again is json` is true. The second import returned the object already stored under `"json"`.
- `marker` is still `"kept"`. The file did not run again, so the attribute you attached is still there.
- `"json"` is in `sys.modules`. That dictionary is the cache.
</details>


## Why it works this way

The first import of a name does three jobs. It walks `sys.path` until it finds a matching file or package. It creates a module object and puts that object in `sys.modules` immediately, before the file runs. Then it executes the file with that module's `__dict__` as the global namespace. The early insert is the whole circular-import story: the other module can import this name and receive the half-built object instead of starting a second copy.

A cache hit skips the file. The second `import` is a dictionary lookup. That is why attaching `marker` survives, and why editing the file on disk changes nothing until something reloads or the process starts over. The cost of the cache is a module object kept for the life of the process, which is small next to the cost of re-running every import. The design reason is initialization that must happen once: registering codecs, opening no second copy of a module's globals, and giving every importer the same object so `is` works.

`sys.path` order is the search. Entry 0 is the program's directory, so a local `json.py` would shadow the standard library. That is a feature for your packages and a bug if you name a file `json.py` by accident. Later entries are the standard library and site-packages. The evidence does not print those paths, because they differ by machine. It checks that the first `site-packages` entry comes after entry 0, so your folder is searched before installed packages.

`from rate import rate` binds your local name to whatever object `rate.rate` is at that moment. In the evidence cell that object is the int `2`. Reload executes the new file inside the same module object, so `rate.rate` becomes `7`. Your local name still points at `2`. Nobody updated your namespace. A function you imported with `from` is a different object from the new `rate.get`. If that old function reads the global `rate`, the read uses the function's `__globals__`, which is still the module dict reload just updated, so `get()` returns `7` even though `get is rate.get` is false. The name in your file is old. The module dict is new. Those are different slots.

A circular import fails when the second module uses a name the first has not assigned yet. The traceback's last line is `AttributeError`, and the message says the module is partially initialized. The fix is to move the import below the names it needs, or to import inside the function that uses them, after both modules have finished. Reloading is the wrong fix. The cycle is in the import graph, not in a stale cache.

`importlib.reload` is for a long-running process where you accept the caveats. It does not rebind `from` imports. It does not re-run the modules that imported this one. A same-second bytecode file can also look newer than the source, so this lesson deletes `__pycache__` before reload. In normal programs you restart the process instead.


In [3]:
import importlib.util

package_index = next(
    index for index, entry in enumerate(sys.path) if "site-packages" in entry
)
print(package_index > 0)

source.write_text("rate = 2\ndef get():\n    return rate\n")
cached = Path(importlib.util.cache_from_source(str(source)))
if cached.exists():
    cached.unlink()
importlib.invalidate_caches()
importlib.reload(rate)
# These assignments are what `from rate import ...` stores.
get = rate.get
copied_rate = rate.rate

source.write_text("rate = 7\ndef get():\n    return rate\n")
cached = Path(importlib.util.cache_from_source(str(source)))
if cached.exists():
    cached.unlink()
importlib.invalidate_caches()
importlib.reload(rate)
print(rate.rate, copied_rate)
print(get is rate.get)
print(get())


True
7 2
False
7


In [4]:
left_path = folder / "left.py"
right_path = folder / "right.py"
left_path.write_text("import right\nvalue = right.RIGHT\n")
right_path.write_text("import left\nRIGHT = left.value + '-R'\n")
for name in ("left", "right"):
    sys.modules.pop(name, None)
importlib.invalidate_caches()
try:
    importlib.import_module("left")
except AttributeError as error:
    print(type(error).__name__)
    print("circular import" in str(error))


AttributeError
True


## When to use it

Import at the top of the file when the dependency is real and not a cycle. Import inside a function when the import would be circular, or when the dependency is optional and most runs should not pay for it. Restart the process when you change a module. Use `reload` only in a long-running tool where you know every `from` import that must be redone.

Do not name your file after a standard library module. `sys.path` searches your folder first. Do not use `from module import name` for a value you expect reload to update. Keep the module object and read `module.name`.

A package is a module too. Its `__init__.py` runs once, on the first import of the package, and the same cache rules apply. Circular imports between submodules fail the same way.


## Common mistake

After reload, the name you bound with `from rate import rate` still holds the old integer. `rate.rate` on the module is the new one. Nothing raises. A check that prints only the copied name stays on the old number.


In [5]:
stale = rate.rate
source.write_text("rate = 8\n")
cached = Path(importlib.util.cache_from_source(str(source)))
if cached.exists():
    cached.unlink()
importlib.invalidate_caches()
importlib.reload(rate)
print(stale)
print(rate.rate)


7
8


`stale` prints `7`, the integer copied before the file became `8`. `rate.rate` prints `8`. The fix is to read `rate.rate` after reload, or to run `from rate import rate` again. Reload will not walk your other modules and update them.


In [6]:
importlib.reload(rate)
fresh = rate.rate
print(fresh)
print(fresh == rate.rate)


8
True


## Try it

Move `value = right.RIGHT` in `left.py` to a function that runs only after both imports have finished. Why does `import left` then succeed, and which name is still missing if `right.py` calls `left.value` at import time?


## What this explains

- [Imports](../../course/13-modules/01-imports-lesson.ipynb): `import` and `from` bind names. The cache is why a second import does not repeat top-level work.
- [Program or library](../../course/13-modules/02-main-lesson.ipynb): the file runs on import. `__name__ == "__main__"` is how a module skips work when it was imported rather than run.
- [Packages](../../course/13-modules/03-packages-lesson.ipynb): a package is a module object too. Its `__init__.py` runs once and then lives in `sys.modules`.
- [Attribute lookup](../08-attributes/01-lookup-lesson.ipynb): `module.name` is a lookup on the module object. After `from`, your local name is a separate slot.


## Recap

- `sys.modules` remembers the module object. A second import does not run the file.
- `from` copies the current object into your namespace. `reload` updates the module, not those copies.
- A circular import sees a half-initialized module because the cache entry is created before the file finishes.

Next: [Source to bytecode](../10-bytecode/01-bytecode-lesson.ipynb).
